# Dự báo Doanh số Thương mại Điện tử
## Giai đoạn 1 — Nền tảng Đánh giá & Chuẩn hóa
---

## Import thư viện

In [29]:
import numpy as np
import pandas as pd

---
## Giai đoạn 1A — Evaluation Metrics
Cung cấp thước đo để đánh giá độ lệch giữa doanh số dự báo `ŷ` và doanh số thực tế `y`.

In [30]:
def calculate_mse(y_true, y_pred):
    """Mean Squared Error — dùng làm Loss Function"""
    n = len(y_true)
    return (1 / n) * np.sum((y_true - y_pred) ** 2)

def calculate_rmse(y_true, y_pred):
    """Root Mean Squared Error — đưa sai số về cùng đơn vị gốc"""
    return np.sqrt(calculate_mse(y_true, y_pred))

def calculate_mae(y_true, y_pred):
    """Mean Absolute Error — ít nhạy cảm với outlier hơn MSE"""
    n = len(y_true)
    return (1 / n) * np.sum(np.abs(y_true - y_pred))

def calculate_r2(y_true, y_pred):
    """R-squared — càng gần 1 càng tốt, âm = tệ hơn dự báo bằng mean"""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

print("Đã định nghĩa xong các hàm Evaluation Metrics")

Đã định nghĩa xong các hàm Evaluation Metrics


In [31]:
# Kiểm tra nhanh Evaluation Metrics
# y_pred lệch đều ±10 so với y_true → MSE = 100, RMSE = MAE = 10
# R² tính tay: ss_res=500, ss_tot=100000 → R²=0.9950 (không phải 0.9975)
y_true_check = np.array([100, 200, 300, 400, 500], dtype=float)
y_pred_check = np.array([110, 190, 310, 390, 510], dtype=float)

mse_val  = calculate_mse(y_true_check, y_pred_check)
rmse_val = calculate_rmse(y_true_check, y_pred_check)
mae_val  = calculate_mae(y_true_check, y_pred_check)
r2_val   = calculate_r2(y_true_check, y_pred_check)

print("── Kiểm tra Evaluation Metrics ──")
print(f"MSE  : {mse_val:.2f}   (kỳ vọng: 100.00)")
print(f"RMSE : {rmse_val:.2f}   (kỳ vọng: 10.00)")
print(f"MAE  : {mae_val:.2f}   (kỳ vọng: 10.00)")
print(f"R²   : {r2_val:.4f}  (kỳ vọng: 0.9950)")

# Assert tự động để phát hiện sai sót
assert abs(mse_val  - 100.0)  < 1e-6, f"MSE sai: {mse_val}"
assert abs(rmse_val - 10.0)   < 1e-6, f"RMSE sai: {rmse_val}"
assert abs(mae_val  - 10.0)   < 1e-6, f"MAE sai: {mae_val}"
assert abs(r2_val   - 0.9950) < 1e-3, f"R² sai: {r2_val}"
print("\nTất cả assert Evaluation Metrics PASSED.")

── Kiểm tra Evaluation Metrics ──
MSE  : 100.00   (kỳ vọng: 100.00)
RMSE : 10.00   (kỳ vọng: 10.00)
MAE  : 10.00   (kỳ vọng: 10.00)
R²   : 0.9950  (kỳ vọng: 0.9950)

Tất cả assert Evaluation Metrics PASSED.


---
## Giai đoạn 1B — CustomStandardScaler
Chuẩn hóa các đặc trưng về cùng hệ quy chiếu bằng Z-score: `Z = (X - µ) / σ`

> **Quan trọng:** Chỉ `fit()` trên tập **Train**. Tập Test chỉ được `transform()` bằng µ/σ đã học từ Train — tránh **data leakage**.

In [32]:
class CustomStandardScaler:
    def __init__(self):
        self.mean_ = None  # µ — lưu lại sau fit()
        self.std_  = None  # σ — lưu lại sau fit()

    def fit(self, X):
        """Tính µ và σ từ tập Train"""
        self.mean_ = np.mean(X, axis=0)
        self.std_  = np.std(X, axis=0)
        # Tránh chia cho 0 nếu feature có std = 0
        self.std_[self.std_ == 0] = 1
        return self

    def transform(self, X):
        """Áp dụng Z-score: Z = (X - µ) / σ"""
        if self.mean_ is None:
            raise RuntimeError("Cần gọi fit() trước khi transform()")
        return (X - self.mean_) / self.std_

    def fit_transform(self, X):
        """Gọi fit() rồi transform() — tiện cho tập Train"""
        return self.fit(X).transform(X)

print("Đã định nghĩa xong CustomStandardScaler")

Đã định nghĩa xong CustomStandardScaler


In [33]:
# Kiểm tra nhanh CustomStandardScaler
X_sample = np.array([[1000, 5], [2000, 10], [3000, 15], [4000, 20]], dtype=float)

scaler_test = CustomStandardScaler()
X_scaled_test = scaler_test.fit_transform(X_sample)

mean_after = X_scaled_test.mean(axis=0).round(6)
std_after  = X_scaled_test.std(axis=0).round(6)

print("── Kiểm tra CustomStandardScaler ──")
print(f"Mean sau chuẩn hóa (kỳ vọng ≈ 0): {mean_after}")
print(f"Std  sau chuẩn hóa (kỳ vọng ≈ 1): {std_after}")

# Assert tự động
assert np.allclose(mean_after, 0, atol=1e-5), f"Mean không ≈ 0: {mean_after}"
assert np.allclose(std_after,  1, atol=1e-5), f"Std không ≈ 1: {std_after}"

# Kiểm tra transform() trên tập khác chỉ dùng µ/σ từ train (không fit lại)
X_new = np.array([[500, 2], [5000, 25]], dtype=float)
X_new_scaled = scaler_test.transform(X_new)
assert X_new_scaled.shape == (2, 2), "Shape sau transform sai"

print("\nTất cả assert CustomStandardScaler PASSED.")

── Kiểm tra CustomStandardScaler ──
Mean sau chuẩn hóa (kỳ vọng ≈ 0): [0. 0.]
Std  sau chuẩn hóa (kỳ vọng ≈ 1): [1. 1.]

Tất cả assert CustomStandardScaler PASSED.


---
## Áp dụng lên dữ liệu thực tế

In [34]:
# Load dữ liệu — thay đường dẫn cho đúng với máy của bạn
X_train = pd.read_csv("../dataset_ready/X_train.csv").values.astype(float)
X_test  = pd.read_csv("../dataset_ready/X_test.csv").values.astype(float)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")

# Kiểm tra NaN
assert not np.isnan(X_train).any(), "X_train có giá trị NaN!"
assert not np.isnan(X_test).any(),  "X_test có giá trị NaN!"
print("\nDữ liệu hợp lệ, không có NaN")

X_train : (487, 9)
X_test  : (122, 9)

Dữ liệu hợp lệ, không có NaN


In [35]:
# Chuẩn hóa
scaler     = CustomStandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # fit chỉ trên train
X_test_sc  = scaler.transform(X_test)       # transform test bằng µ/σ của train

print("── Thông số Scaler ──")
print(f"µ (mean): {scaler.mean_}")
print(f"σ (std) : {scaler.std_}")

# Assert kiểm tra chất lượng chuẩn hóa trên dữ liệu thực
train_mean_after = X_train_sc.mean(axis=0)
train_std_after  = X_train_sc.std(axis=0)

assert np.allclose(train_mean_after, 0, atol=1e-6), \
    f"X_train_sc mean không ≈ 0: {train_mean_after}"
assert np.allclose(train_std_after,  1, atol=1e-6), \
    f"X_train_sc std không ≈ 1: {train_std_after}"

print("\nChuẩn hóa hoàn tất — X_train_sc và X_test_sc sẵn sàng cho Giai đoạn 2")

── Thông số Scaler ──
µ (mean): [1.39088812e+02 3.23350290e-01 2.03967325e+04 2.00093339e+04
 2.02341362e+04 3.01437372e+00 2.87474333e-01 1.57166324e+01
 5.62628337e+00]
σ (std) : [2.98391866e+01 6.81736992e-02 1.20832516e+04 1.21334640e+04
 1.01121224e+04 2.00148784e+00 4.52584623e-01 8.71413392e+00
 3.51295156e+00]

Chuẩn hóa hoàn tất — X_train_sc và X_test_sc sẵn sàng cho Giai đoạn 2


Class DanhGiaMoHinh

> **Đây là Class gốc.** Tất cả thuật toán ở các giai đoạn đều **bắt buộc** gọi class này vào để đánh giá — không ai được code lại hàm tính toán sai số.

In [36]:
class DanhGiaMoHinh:
    """
    Class đánh giá mô hình dùng chung cho toàn bộ pipeline.
    Gọi lại các hàm metric đã định nghĩa ở 1A — không code lại.
    Inject vào tất cả các thuật toán ở Bước 2, 3, 4, 5.
    """

    def __init__(self):
        pass

    def evaluate_all(self, y_true, y_pred, model_name="Model"):
        """Tính và in toàn bộ metrics. Trả về dict kết quả."""
        y_true = np.asarray(y_true).ravel()
        y_pred = np.asarray(y_pred).ravel()
        if y_true.shape != y_pred.shape:
            raise ValueError("y_true và y_pred phải cùng shape")

        results = {
            "MSE":  calculate_mse(y_true, y_pred),
            "RMSE": calculate_rmse(y_true, y_pred),
            "MAE":  calculate_mae(y_true, y_pred),
            "R2":   calculate_r2(y_true, y_pred),
        }

        print(f"── Đánh giá {model_name} ──")
        print(f"MSE  : {results['MSE']:.4f}")
        print(f"RMSE : {results['RMSE']:.4f}")
        print(f"MAE  : {results['MAE']:.4f}")
        print(f"R²   : {results['R2']:.4f}")
        return results


# Khởi tạo 1 lần duy nhất — dùng xuyên suốt toàn bộ project
my_evaluator = DanhGiaMoHinh()
print("Đã khởi tạo my_evaluator — sẵn sàng inject vào các thuật toán.")

Đã khởi tạo my_evaluator — sẵn sàng inject vào các thuật toán.


---
## Bước 2 — CustomLinearRegression

- Dùng **Ordinary Least Squares** (Normal Equation) qua `np.linalg.pinv`.
- Nhận `evaluator` là object của `DanhGiaMoHinh` — **không tự tính metric bên trong**.

In [37]:
class CustomLinearRegression:
    """
    Hồi quy tuyến tính bằng Normal Equation.
    Nhận evaluator (DanhGiaMoHinh) qua dependency injection.
    """

    def __init__(self, evaluator):
        self.weights   = None
        self.bias      = None
        self.evaluator = evaluator  # inject từ Bước 1

    def fit(self, X, y):
        """Huấn luyện bằng Normal Equation: θ = (XᵀX)⁻¹ Xᵀy"""
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).ravel()

        n_samples, n_features = X.shape
        X_design = np.hstack([np.ones((n_samples, 1)), X])  # thêm cột bias

        params       = np.linalg.pinv(X_design.T @ X_design) @ X_design.T @ y
        self.bias    = params[0]
        self.weights = params[1:]
        return self

    def predict(self, X):
        """Dự báo: ŷ = Xw + b"""
        if self.weights is None or self.bias is None:
            raise RuntimeError("Cần huấn luyện model trước khi predict()")
        X = np.asarray(X, dtype=float)
        return X @ self.weights + self.bias

    def evaluate(self, X_test, y_test, model_name="Linear Regression"):
        """Dự báo rồi gọi evaluator từ Bước 1 để in kết quả"""
        y_pred = self.predict(X_test)
        return self.evaluator.evaluate_all(y_test, y_pred, model_name=model_name)


print("Đã định nghĩa xong CustomLinearRegression")

Đã định nghĩa xong CustomLinearRegression


In [38]:
# Load target values từ dataset_ready
y_train = pd.read_csv("../dataset_ready/y_train.csv").values.ravel().astype(float)
y_test  = pd.read_csv("../dataset_ready/y_test.csv").values.ravel().astype(float)

# Kiểm tra NaN
assert not np.isnan(y_train).any(), "y_train có giá trị NaN!"
assert not np.isnan(y_test).any(),  "y_test có giá trị NaN!"
print(f"y_train: {y_train.shape}  |  y_test: {y_test.shape}")

# Huấn luyện và đánh giá
lr_model = CustomLinearRegression(evaluator=my_evaluator)
lr_model.fit(X_train_sc, y_train)
lr_results = lr_model.evaluate(X_test_sc, y_test)

# Kiểm tra sanity: R² phải > 0 (tốt hơn dự báo bằng mean)
assert lr_results["R2"] > 0, f"R² âm — mô hình tệ hơn baseline mean: {lr_results['R2']:.4f}"
print(f"\nSanity check R² > 0: PASSED ({lr_results['R2']:.4f})")

y_train: (487,)  |  y_test: (122,)
── Đánh giá Linear Regression ──
MSE  : 39675593.1754
RMSE : 6298.8565
MAE  : 4996.0057
R²   : 0.6860

Sanity check R² > 0: PASSED (0.6860)
